In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
import os

# Custom function to preprocess images and ensure they are RGB
def preprocess_image_rgb(image):
    print("Original shape:", image.shape)
    print("Original dtype:", image.dtype)
    if image.shape[-1] != 3:  # Check if the last dimension is not 3
        raise ValueError(f"Expected image with 3 channels, but got shape {image.shape}")
    image = image.astype('uint8')  # Convert to uint8
    image = Image.fromarray(image)  # Convert to PIL Image
    image = image.convert('RGB')   # Convert to RGB
    return img_to_array(image)

# Set paths for training and testing datasets
train_dir = 'train'
test_dir = 'test'

# Data generators
train_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: preprocess_input(preprocess_image_rgb(x)),
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: preprocess_input(preprocess_image_rgb(x))
)

# Load data from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

# Load MobileNet model with pre-trained weights
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Freeze the base model
base_model.trainable = False

# Add custom layers
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation='softmax')
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Display class indices
print("Classes and their corresponding indices:")
print(train_generator.class_indices)

# Display sorted class names
class_names = sorted(train_generator.class_indices.keys())
print("Classes in alphabetical order:")
print(class_names)

# Model summary
model.summary()

# Train the model
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator
)

# Save the model
model.save('mobilenet_model.h5')

# Evaluate the model
loss, accuracy = model.evaluate(test_generator)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")


Found 13368 images belonging to 7 classes.
Found 681 images belonging to 7 classes.
Classes and their corresponding indices:
{'Oral_cancer': 0, 'calculus': 1, 'caries': 2, 'gingivitis': 3, 'hypodontia': 4, 'mouth_ulcer': 5, 'tooth_discoloration': 6}
Classes in alphabetical order:
['Oral_cancer', 'calculus', 'caries', 'gingivitis', 'hypodontia', 'mouth_ulcer', 'tooth_discoloration']


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenet_1.00_128 (Functional)      │ (None, 4, 4, 1024)          │       3,228,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 1024)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │         262,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 7)                   │           1,799 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,493,063 (13.32 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 3,228,864 (12.32 MB)

Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3

C:\Python312\Lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
 86/418 ━━━━━━━━━━━━━━━━━━━━ 2:53 521ms/step - accuracy: 0.1877 - loss: 3.3471Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Original shape: (128, 128, 3)
Original dtype: float32
Orig

In [2]:
!pip install visualkeras

In [6]:
import visualkeras
from tensorflow.keras.applications import MobileNetV2

# Load the MobileNetV2 model
model = MobileNetV2(weights=None, input_shape=(224, 224, 3), classes=1000)

# Visualize the architecture using visualkeras
visualkeras.layered_view(
    model,
    to_file="mobilenetv2_architecture.png",  # Save the diagram to a file
    legend=True  # Include a legend to explain the colors/shapes
).show()  # Display the architecture diagram
